# Argopy accessor

An argopy dataset is matched in one call, with the `floatmatcher` extension of its `.argo` accessor: `ds.argo.floatmatcher.match(product=..., method=...)`.

In [ ]:
import numpy as np
from argopy import DataFetcher
import floatmatcher  # noqa: F401  (registers ds.argo.floatmatcher)

### Argo data

In [ ]:
argo = DataFetcher().region([-28, -25, 48, 50, 0, 50, "2018-01", "2018-02"]).data  # N_POINTS layout
argo

### Match

In [ ]:
product = dict(type="era5-hourly", source="local", selected_variables=["sst"],
               path="/runtime/data/era5-hourly",  # adapt to your archive
               pattern="{year}/{month:02d}/era5_single-levels_{year}{month:02d}{day:02d}.nc")

argo.argo.floatmatcher.match(product=product,
                             method=dict(type="nearest", radius=50, time_gap=np.timedelta64(12, "h")))
# the dataset comes back in its layout, with sst_coloc

### Several neighbors

With `k_nearest` or `n_frames` above 1, each point has several values: the `k_space` and `n_time` dimensions. A dataset in points layout (`N_POINTS`) cannot take them back: without an option, `match` raises an error.

In [ ]:
method = dict(type="nearest", radius=50, time_gap=np.timedelta64(12, "h"),
              k_nearest=2, n_frames=2)

try:
    argo.argo.floatmatcher.match(product=product, method=method)  # no option
except ValueError as error:
    print(error)

`neigh_to_vars=True`: one variable per neighbor along `N_POINTS`, `s0` the nearest node, `t0` the nearest time step.

In [ ]:
argo.argo.floatmatcher.match(product=product, method=method, neigh_to_vars=True)

`original_shape=False`: back in profiles (`N_PROF`, `N_LEVELS`), with the `k_space` and `n_time` dimensions.

In [ ]:
argo.argo.floatmatcher.match(product=product, method=method, original_shape=False)